In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone 3

In [2]:
!pip install faiss-cpu #install FAISS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 45.5 MB/s eta 0:00:00


In [3]:
import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating knowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating knowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points) *


In [4]:
zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Row 150
row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

correct_option = row_150["answer"]
correct_text = str(row_150[correct_option])

# Zero-shot prediction
result = zs(
    prompt_150,
    candidate_labels=labels_150
)

# Print all predictions
for label, score in zip(result["labels"], result["scores"]):
    print(f"{score:.3f}  {label}")

# Score of ground-truth answer
idx = result["labels"].index(correct_text)

print("\nGround Truth Option :", correct_option)
print("Ground Truth Text:", correct_text)
print("Probability:", round(result["scores"][idx], 3))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

0.384  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.379  The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.093  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."
0.079  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.066  The butterfly effect is the p

2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)? *
Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  

In [5]:
# Prompt embedding
query_embedding = model.encode(
    [prompt_150],
    show_progress_bar=False
)

# Search top 10
k = 10

distances, indices = index.search(
    np.array(query_embedding),
    k
)

retrieved_indices = indices[0]

print("Retrieved KB indices:")
print(retrieved_indices)

# Find rank of correct document
true_index = 150

if true_index in retrieved_indices:

    rank = list(retrieved_indices).index(true_index) + 1

    print("True document rank:", rank)

else:

    print("True document NOT found in top", k)

Retrieved KB indices:
[ 663 1701 1269 1532  576  847 1693 1906  168  150]
True document rank: 10


3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document? 

In [6]:
# Load Cross Encoder
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

# Top-10 documents from FAISS
docs_10 = [kb[i] for i in retrieved_indices]

# Create (query, document) pairs
pairs = [
    [prompt_150, doc]
    for doc in docs_10
]

# Predict relevance scores
ce_scores = cross_encoder.predict(pairs)

# Sort documents by score
sorted_idx = np.argsort(ce_scores)[::-1]

print("Re-ranked Results\n")

for rank, idx in enumerate(sorted_idx, start=1):

    kb_index = retrieved_indices[idx]

    print(
        f"Rank {rank} | "
        f"KB Index = {kb_index} | "
        f"Score = {ce_scores[idx]:.4f}"
    )

# Rank of true document
true_index = 150

reranked_indices = [
    retrieved_indices[i]
    for i in sorted_idx
]

true_rank = reranked_indices.index(true_index) + 1

print("\nTrue Document Rank =", true_rank)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Re-ranked Results

Rank 1 | KB Index = 150 | Score = 4.7585
Rank 2 | KB Index = 1906 | Score = 4.7526
Rank 3 | KB Index = 847 | Score = 4.7526
Rank 4 | KB Index = 1693 | Score = 4.7526
Rank 5 | KB Index = 1269 | Score = 4.7375
Rank 6 | KB Index = 1532 | Score = 4.7375
Rank 7 | KB Index = 168 | Score = 4.7072
Rank 8 | KB Index = 576 | Score = 4.6870
Rank 9 | KB Index = 1701 | Score = 4.6602
Rank 10 | KB Index = 663 | Score = 4.6602

True Document Rank = 1


4.  Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate? *


In [7]:
row_42 = train.iloc[42]

prompt_42 = str(row_42["prompt"])

# Encode prompt
query_embedding = model.encode(
    [prompt_42],
    show_progress_bar=False
)

# Retrieve top 5
k = 5

_, indices = index.search(
    np.array(query_embedding),
    k
)

docs = [
    kb[i]
    for i in indices[0]
]

# Build Context
context = " ".join(docs)

rag_string = (
    "Context: "
    + context
    + " Question: "
    + prompt_42
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

tokens = tokenizer(
    rag_string,
    truncation=False
)

print("Total Tokens =", len(tokens["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Total Tokens = 216


5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places). *


In [8]:
# Ground-truth document
true_document = kb[150]

rag_prompt = (
    "Context: "
    + true_document
    + " Question: "
    + prompt_150
)

# Zero-shot prediction
result = zs(
    rag_prompt,
    candidate_labels=labels_150
)

# Ground truth answer text
correct_text = str(
    row_150[
        row_150["answer"]
    ]
)

# Probability
idx = result["labels"].index(
    correct_text
)

print(
    "Probability =",
    round(result["scores"][idx], 3)
)

print()

for label, score in zip(
    result["labels"],
    result["scores"]
):
    print(
        f"{score:.3f}",
        label
    )

Probability = 0.989

0.989 The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.004 The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.003 The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.002 The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."
0.002 The butterfly

6.  What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places). *


In [9]:
row = train.iloc[150]

prompt = str(row["prompt"])

wrong_doc = kb[999]

labels = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

correct = str(row[row["answer"]])

rag = f"Context: {wrong_doc} Question: {prompt}"

result = zs(rag, candidate_labels=labels)

prob = dict(zip(result["labels"], result["scores"]))[correct]

print(round(prob,3))

0.529


7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place). *


In [10]:
hits = 0

for idx in range(100):

    row = train.iloc[idx]

    prompt = str(row["prompt"])

    correct_doc = str(row[row["answer"]])

    query_embedding = model.encode([prompt], show_progress_bar=False)

    _, I = index.search(query_embedding, 5)

    retrieved_docs = [kb[i] for i in I[0]]

    if correct_doc in retrieved_docs:
        hits += 1

hit_rate = hits / 100 * 100

print("Hit Rate:", round(hit_rate, 1))

Hit Rate: 73.0


8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score:
 Look at the probability scores output by the model. Rank the options 
from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [11]:
import numpy as np

def map_at_3(actual, predicted):
    """
    actual: correct answer letter (e.g. 'A')
    predicted: list of predicted letters in ranked order
    """
    for i, p in enumerate(predicted[:3]):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0

scores = []

for idx in range(20):

    row = train.iloc[idx]

    prompt = str(row["prompt"])

    options = {
        "A": str(row["A"]),
        "B": str(row["B"]),
        "C": str(row["C"]),
        "D": str(row["D"]),
        "E": str(row["E"])
    }

    #Retrieval
    query_embedding = model.encode([prompt], show_progress_bar=False)

    _, I = index.search(query_embedding, 5)

    retrieved_docs = [kb[i] for i in I[0]]

    # Reranking 
    pairs = [[prompt, doc] for doc in retrieved_docs]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = retrieved_docs[np.argmax(ce_scores)]

    # RAG
    rag_prompt = f"Context: {best_doc} Question: {prompt}"

    candidate_labels = list(options.values())

    result = zs(
        rag_prompt,
        candidate_labels=candidate_labels,
        multi_label=False
    )

    # Convert option text back to option letters
    predicted_letters = []

    for label in result["labels"]:
        for letter, text in options.items():
            if label == text:
                predicted_letters.append(letter)
                break

    predicted_letters = predicted_letters[:3]

    score = map_at_3(row["answer"], predicted_letters)

    scores.append(score)

print("MAP@3 =", round(np.mean(scores), 3))

MAP@3 = 0.975
